In [1]:
import numpy as np
from pyscf import gto, scf, mp, cc, lo
from pyscf.data import elements

a = 1.20577 # bond length in a cluster
d = 4 # distance between each cluster
unit = 'A' # unit of length
na = 2 # size of a cluster (monomer)
nc = 2 # set as integer multiple of monomers
spin = 2 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'O'
basis = 'sto6g'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms,
            basis=basis,
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=spin*nc,
            max_memory=4000,
            )

mf = scf.UHF(mol).density_fit()
mf.kernel()

stable = False
while not stable:
    print(f'mean-field stability test')
    if not stable:
        mo_i, _, stable,_ = mf.stability(return_status=True)
        dm = mf.make_rdm1(mo_i,mf.mo_occ)
        mf.kernel(dm0=dm)
    elif stable:
        print(f'UHF Energy: {mf.e_tot}, stability {stable}')
        break

mymp = mp.MP2(mf).set_frozen()
mymp.kernel()

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()

print(f"HF   : {mf.e_tot}")
print(f"MP2  : {mymp.e_tot}")
print(f"CCSD : {mycc.e_tot}")

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-35-generic', version='#35~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue May 26 19:30:42 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Thu Jul  9 17:59:26 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 4
[INPUT] num. electrons = 32
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 4
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = A
[INPUT] Symbol           X                Y                Z      uni

In [2]:
print(f"HF per M   : {mf.e_tot/nc}")
print(f"MP2 per M  : {mymp.e_tot/nc}")
print(f"CCSD per M : {mycc.e_tot/nc}")

HF per M   : -149.0508159109543
MP2 per M  : -149.12980029401535
CCSD per M : -149.1584774543587


In [3]:
frozen = elements.chemcore(mol)
nocc = (np.count_nonzero(mf.mo_occ[0]),
        np.count_nonzero(mf.mo_occ[1]))
print(nocc)
lo_occ_a = lo.PipekMezey(mol, mf.mo_coeff[0][:,frozen:nocc[0]]).kernel()
lo_vir_a = lo.PipekMezey(mol, mf.mo_coeff[0][:,nocc[0]:]).kernel()
lo_occ_b = lo.PipekMezey(mol, mf.mo_coeff[1][:,frozen:nocc[1]]).kernel()
lo_vir_b = lo.PipekMezey(mol, mf.mo_coeff[1][:,nocc[1]:]).kernel()
print(lo_occ_a.shape, lo_vir_a.shape)
print(lo_occ_b.shape, lo_vir_b.shape)
lo_coeff = (np.hstack((mf.mo_coeff[0][:,:frozen],lo_occ_a,lo_vir_a)),
            np.hstack((mf.mo_coeff[1][:,:frozen],lo_occ_b,lo_vir_b)))
dm0 = mf.make_rdm1(lo_coeff, mf.mo_occ)
mf1 = scf.UHF(mol).density_fit()
mf1.kernel(dm0=dm0)

(np.int64(18), np.int64(14))


******** <class 'pyscf.lo.pipek.PipekMezey'> ********
conv_tol = 1e-06
conv_tol_grad = None
max_cycle = 100
max_stepsize = 0.05
max_iters = 20
kf_interval = 5
kf_trust_region = 5
ah_start_tol = 1000000000.0
ah_start_cycle = 1
ah_level_shift = 0
ah_conv_tol = 1e-12
ah_lindep = 1e-14
ah_max_cycle = 40
ah_trust_region = 3
init_guess = atomic
pop_method = meta_lowdin
Set conv_tol_grad to 0.000316228
macro= 1  f(x)= 12.977926359298  delta_f= 12.9779  |g|= 0.365133  2 KF 7 Hx
macro= 2  f(x)= 12.999979949906  delta_f= 0.0220536  |g|= 0.00250105  2 KF 4 Hx
macro= 3  f(x)= 12.999979963952  delta_f= 1.40461e-08  |g|= 0.000259205  1 KF 2 Hx
macro X = 3  f(x)= 12.999979963952  |g|= 0.000259205  6 intor 5 KF 13 Hx


******** <class 'pyscf.lo.pipek.PipekMezey'> ********
conv_tol = 1e-06
conv_tol_grad = None
max_cycle = 100
max_stepsize = 0.05
max_iters = 20
kf_interval = 5
kf_trust_region = 5
ah_start_tol = 1000000000.0
ah_start_cycle = 1
ah_level_shift = 0
ah_conv_tol

np.float64(-298.10163182191263)

In [4]:
def check_span(mo1, s1e, mo2, thresh = 1e-10):
    '''
    check if mo1 and mo2 span each other
    return (mo1 span mo2),  (mo2 span mo1)
    '''
    olp11 = mo1.T.conj() @ s1e @ mo1
    olp12 = mo1.T.conj() @ s1e @ mo2
    olp22 = mo2.T.conj() @ s1e @ mo2

    span12 = np.abs(olp12.T.conj() @ olp12 - olp22).max() < thresh
    span21 = np.abs(olp12 @ olp12.T.conj() - olp11).max() < thresh
    # span12 = np.abs(olp12.conj().T @ np.linalg.solve(olp11, olp12) - olp22).max() < thresh
    # span21 = np.abs(olp12 @ np.linalg.solve(olp22, olp12.conj().T) - olp11).max() < thresh

    return span12, span21

mo1 = lo_coeff[0][:,frozen:nocc[0]]
mo2 = mf.mo_coeff[0][:,frozen:nocc[0]]
s1e = mf.get_ovlp()
print(check_span(mo1, s1e, mo2, thresh = 1e-10))

(np.True_, np.True_)


In [5]:
mf1.mo_coeff = lo_coeff

mymp1 = mp.MP2(mf1).set_frozen()
mymp1.kernel()

mycc1 = cc.CCSD(mf1).set_frozen()
mycc1.kernel()


print(f"HF   : {mf.e_tot:.10f}  {mf1.e_tot:.10f}")
print(f"MP2  : {mymp.e_tot:.10f} {mymp1.e_tot:.10f}")
print(f"CCSD : {mycc.e_tot:.10f}  {mycc1.e_tot:.10f}")


******** <class 'pyscf.mp.dfump2.DFUMP2'> ********
nocc = (np.int64(14), np.int64(10)), nmo = (16, 16)
frozen orbitals 4
max_memory 4000 MB (current use 207 MB)
E(DFUMP2) = -298.229324193276  E_corr = -0.127692371363769
E(SCS-DFUMP2) = -298.203690965962  E_corr = -0.102059144049233
E_corr(same-spin) = -0.0590442710622576
E_corr(oppo-spin) = -0.0686481003015114

******** <class 'pyscf.cc.dfuccsd.UCCSD'> ********
CC2 = 0
CCSD nocc = (np.int64(14), np.int64(10)), nmo = (16, 16)
frozen orbitals 4
max_cycle = 50
direct = 0
conv_tol = 1e-07
conv_tol_normt = 1e-06
diis_space = 6
diis_start_cycle = 0
diis_start_energy_diff = 1e+09
max_memory 4000 MB (current use 207 MB)
Init t2, MP2 energy = -0.142803923656562
Init E_corr(UCCSD) = -0.142803923657637
cycle = 1  E_corr(UCCSD) = -0.172941119839078  dE = -0.0301371962  norm(t1,t2) = 0.10704
cycle = 2  E_corr(UCCSD) = -0.187282021956336  dE = -0.0143409021  norm(t1,t2) = 0.0803114
cycle = 3  E_corr(UCCSD) = -0.219793135466409  dE = -0.0325111135  

In [6]:
print(mf.mo_energy)

[[-20.70396372 -20.70396356 -20.70334336 -20.70334332  -1.63086885
   -1.63085675  -1.1014186   -1.10126101  -0.72355048  -0.72354782
   -0.72353166  -0.72352982  -0.61463375  -0.61425791  -0.4191414
   -0.41913961  -0.41910908  -0.41910828   0.67243811   0.67283172]
 [-20.67215728 -20.67215716 -20.67147901 -20.67147899  -1.49662323
   -1.49661387  -0.91026383  -0.91005994  -0.56183933  -0.56144069
   -0.46748984  -0.4674879   -0.46735093  -0.46734573   0.27976587
    0.27976908   0.27990924   0.27991643   0.75886866   0.75923086]]


In [12]:
mo = mf.mo_coeff
fock_ao = mf.get_fock()
fock_mo = (mo[0][:,frozen:].T @ fock_ao[0] @ mo[0][:,frozen:], mo[1][:,frozen:].T @ fock_ao[1] @ mo[1][:,frozen:])
print((fock_mo[0] - np.diag(fock_mo[0].diagonal())).max())
print((fock_mo[1] - np.diag(fock_mo[1].diagonal())).max())
print(fock_mo[0].diagonal())
print(fock_mo[1].diagonal())

5.484648538112147e-08
4.335373317072691e-07
[-1.63086885 -1.63085675 -1.1014186  -1.10126101 -0.72355048 -0.72354782
 -0.72353166 -0.72352982 -0.61463375 -0.61425792 -0.4191414  -0.41913961
 -0.41910909 -0.41910829  0.67243811  0.67283172]
[-1.49662323 -1.49661387 -0.91026383 -0.91005994 -0.56183933 -0.5614407
 -0.46748982 -0.46748788 -0.46735093 -0.46734572  0.27976586  0.27976908
  0.27990922  0.27991641  0.75886866  0.75923086]


In [203]:
import jax
jax.config.update("jax_enable_x64", True)
import opt_einsum as oe

In [8]:
from afqmc import integral
integral.prep_integral(mycc, chol_cut=1e-6)


Preparing AFQMC calculation
Calculating Cholesky integrals
Find Density Fit Teonsers in MF object
Integrals will be built by DF Tensors
Building JK matrix
Alpha Cholesky shape: (86, 16, 16) 
 Beta Cholesky shape: (86, 16, 16) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [14 10]
Number of basis functions:  16
Number of Cholesky vectors: 86


In [9]:
options =  {'n_blocks': 1000,
            'n_walkers': 300,
            'max_memory': 8000,
            'seed': 17,
            'trial': 'upt2ccsd_bar',
            'mix_precision': False,
            }

In [10]:
import time

import numpy as np

from afqmc import config, prep, sampling

from functools import partial

print = partial(print, flush=True)
init_time = time.time()

prep.print_start()
config.setup_jax()

ham_data, ham, prop, trial, wave_data, sampler, options = prep.init_afqmc(options)

wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)
h0 = ham_data['h0']

prop_data = prep.init_hf_prop_data(trial, wave_data, ham_data, options)

init_e = prop_data["e_estimate"]
print(f"AFQMC Init energy : {init_e}")
print(f"PYSCF HF Energy :   {mf.e_tot}")


    ________                     _____                    
    ___  __ \___  __________________(_)_____________ _    
    __  /_/ /  / / /_  __ \_  __ \_  /__  __ \_  __ `/    
    _  _, _// /_/ /_  / / /  / / /  / _  / / /  /_/ /     
    /_/ |_| \__,_/ /_/ /_//_/ /_//_/  /_/ /_/_\__, /      
                                             /____/       
    _____________________________  ___________            
    ___    |__  ____/_  __ \__   |/  /_  ____/            
    __  /| |_  /_   _  / / /_  /|_/ /_  /                 
    _  ___ |  __/   / /_/ /_  /  / / / /___               
    /_/  |_/_/      \___\_\/_/  /_/  \____/               

Hostname:     sharmagroup-rn
System:       Linux
Node:         sharmagroup-rn
Release:      6.17.0-35-generic
Machine:      x86_64
Processor:    x86_64
JAX backend:  GPU
JAX devices:  [CudaDevice(id=0)]
Device kind:  NVIDIA GeForce RTX 5060 Ti
Platform:     gpu

QMC Parameters
n_blocks        -       1000
n_walkers       -        300
max_memory   

In [11]:
from jax import jit, lax
from jax import numpy as jnp

from afqmc import slater_tools

from functools import partial


In [36]:
walker_init_a = prop_data["walkers"][0][0]
walker_init_b = prop_data["walkers"][1][0]

norb_a, nocc_a = walker_init_a.shape
norb_b, nocc_b = walker_init_b.shape
norb = (norb_a, norb_b)
nocc = (nocc_a, nocc_b)

walker_init = (walker_init_a, walker_init_b)

walker = (jnp.array(np.random.rand(*walker_init_a.shape)), 
          jnp.array(np.random.rand(*walker_init_b.shape)))

bra = (jnp.eye(norb_a)[:,:nocc_a], jnp.eye(norb_b)[:,:nocc_b])

h0 = ham_data["h0"]
h1 = ham_data["h1"]
chol = (ham_data["chol"][0].reshape(-1, norb_a, norb_a),
        ham_data["chol"][1].reshape(-1, norb_b, norb_b))

my_fock = integral.get_ufock(nocc, h1, chol)
print((my_fock[0]-fock_mo[0]).max(), (my_fock[1]-fock_mo[1]).max())

ene_1 = slater_tools.u_energy(bra, walker, h0, h1, chol)
print(ene_1)

ene_2 = slater_tools.u_energy_corr(bra, walker, fock_mo, chol)
print(ene_2, mf.e_tot + ene_2)

ene_3 = slater_tools.u_energy_corr(bra, walker, my_fock, chol)
print(ene_3, mf.e_tot + ene_3)

3.340839220822289e-07 3.163891693080956e-07
-301.88610732990867
-3.7844769846353024 -301.8861088065439
-3.784477386732729 -301.8861092086413


In [37]:
from afqmc.lno_afqmc import lno_afqmc, tools
from pyscf.data import elements
iao_coeff, frag_lolist, atm_center = tools.iao_localization(mf1)

mo1 = lo_occ_a
mo2 = iao_coeff[0]
s1e = mf.get_ovlp()
print(check_span(mo1, s1e, mo2, thresh = 1e-10))

mo1 = lo_occ_b
mo2 = iao_coeff[1]
s1e = mf.get_ovlp()
print(check_span(mo1, s1e, mo2, thresh = 1e-10))

(np.False_, np.True_)
(np.False_, np.True_)


In [38]:
nfrag = len(frag_lolist)
print(nfrag)
pfrag = [None] * nfrag
print(len(pfrag))
# <lo|mo>
for frag_idx in range(nfrag):
    frag_orb = (iao_coeff[0][:,frag_lolist[frag_idx][0]],
                iao_coeff[1][:,frag_lolist[frag_idx][1]])
    frag2mo = (frag_orb[0].T.conj() @ s1e @ lo_occ_a, 
               frag_orb[1].T.conj() @ s1e @ lo_occ_b)
    pfrag[frag_idx] = (frag2mo[0].T.conj() @ frag2mo[0],
                       frag2mo[1].T.conj() @ frag2mo[1])

4
4


In [39]:
ene_4 = 0
for p in pfrag:
    ene_4 += slater_tools.u_energy_corr_frag(bra, walker, my_fock, chol, p)
print(ene_4, ene_4-ene_3, mf.e_tot + ene_4)

-3.7844773867332897 -5.608846720406291e-13 -301.88610920864187


In [41]:
from jax import scipy as jsp

t1a = jnp.array(wave_data["t1a"])
t1b = jnp.array(wave_data["t1b"])
t2aa = jnp.array(wave_data["t2aa"])
t2ab = jnp.array(wave_data["t2ab"])
t2bb = jnp.array(wave_data["t2bb"])

nocca, nvira = t1a.shape
noccb, nvirb = t1b.shape
norba = nocca + nvira
norbb = noccb + nvirb
t1a_full = np.zeros((norba, norba))
t1a_full[:nocca, nocca:] = t1a
t1b_full = np.zeros((norbb, norbb))
t1b_full[:noccb, noccb:] = t1b

wave_data['exp_t1a']  = jsp.linalg.expm(jnp.array(t1a_full))
wave_data['exp_mt1a'] = jsp.linalg.expm(jnp.array(-t1a_full))
wave_data['exp_t1b']  = jsp.linalg.expm(jnp.array(t1b_full))
wave_data['exp_mt1b'] = jsp.linalg.expm(jnp.array(-t1b_full))

In [42]:
import opt_einsum as oe 
h1bar_a = wave_data['exp_t1a'] @ h1[0] @ wave_data['exp_mt1a']
h1bar_b = wave_data['exp_t1b'] @ h1[1] @ wave_data['exp_mt1b']
h1_bar = (h1bar_a, h1bar_b)
ham_data["h1bar"] = h1_bar
chol_bar_a = oe.contract('pr,grs,sq->gpq', wave_data['exp_t1a'], chol[0], wave_data['exp_mt1a'], backend='jax')
chol_bar_b = oe.contract('pr,grs,sq->gpq', wave_data['exp_t1b'], chol[1], wave_data['exp_mt1b'], backend='jax')
chol_bar = (chol_bar_a, chol_bar_b)
ham_data["chol_bar"] = chol_bar
fock_bar = integral.get_ufock((nocca, noccb), (h1bar_a, h1bar_b), (chol_bar_a, chol_bar_b))

In [51]:
print(mf.e_tot + mycc.energy(t1=mycc.t1, t2=(mycc.t2[0]*0, mycc.t2[1]*0, mycc.t2[2]*0)))

-298.1114061520016


In [50]:
bra_tilde = (wave_data['exp_t1a'].T @ bra[0], wave_data['exp_t1b'].T @ bra[1])

walker_a_bar = wave_data['exp_t1a'] @ walker[0]
walker_b_bar = wave_data['exp_t1b'] @ walker[1]
walker_bar = (walker_a_bar, walker_b_bar)

# o0 = jnp.linalg.det(walker_a[:walker_a.shape[1],:]) \
#             * jnp.linalg.det(walker_b[:walker_b.shape[1],:])

# obar = jnp.linalg.det(walker_a_bar[:walker_a_bar.shape[1], :]) \
#     * jnp.linalg.det(walker_b_bar[:walker_b_bar.shape[1], :])

# t1olp = obar/o0 # <exp(T1)HF|walker>/<HF|walker>

ene5 = slater_tools.u_energy(bra_tilde, bra, h0, h1, chol)
print(ene5)

ene6 = slater_tools.u_energy(bra, walker_bar, h0, h1_bar, chol_bar)
print(ene6)

ecorr_bar = slater_tools.u_energy_corr(bra, walker_bar, fock_bar, chol_bar)
print(ene5 + ecorr_bar)

-298.11140429329384
-298.49977808060305
-298.4997780806031


In [53]:
ene_7 = 0
for p in pfrag:
    ene_7 += slater_tools.u_energy_corr_frag(bra, walker_bar, fock_bar, chol_bar, p)
print(ene_7, ene_7-ecorr_bar, ene5 + ene_7)

-0.38837378730941935 -1.5587531265737198e-13 -298.4997780806033
